# 10. Final figures, claim ledger, and the return bundle

**The question.** Can every headline figure and number be rebuilt from saved results alone, traced to its source, and handed to the communication stage?

**Why a portfolio manager cares.** The executive summary, slides, and talk must say exactly what the evidence says. A claim ledger ties each number to the file and notebook that produced it.

**Learning objectives**

1. Rebuild the final figures from saved results only (no refitting).
2. Export the summary tables.
3. Write the machine-readable claim ledger with a status for every claim.
4. Link each limitation to its evidence.
5. Write the run manifest and package the return bundle.

| | |
|---|---|
| Inputs | `results/**` written by notebooks 01 to 09 |
| Outputs | `reports/figures/fig_*`, `reports/tables/*`, `results/claims/claim_ledger.{csv,json}`, `results/claims/limitations_evidence.csv`, `results/run_manifest.json`, `return_bundle/*.zip` |
| Run after | 08 (and 09 if available) |
| Approximate runtime | under a minute |

> **No model is refit in this notebook.** Every figure reads saved CSV or Parquet files. If a number here disagrees with notebook 08, the saved file is the source of truth and the disagreement is a bug to report.

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import freeze as fz
from mbshedge.claims import ClaimLedger, status_for
rec = fz.require_freeze(cfg, paths)
R = paths.results
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
monthly = pd.read_parquet(paths.processed / "monthly_panel.parquet")
fc_test = pl.load_forecasts(R / "forecasts" / "forecasts_test.csv")
base = pl.load_runs(R / "backtest", "test_base")
main = pd.read_csv(R / "evaluation" / "main_table_test.csv", index_col=0)
boot = pd.read_csv(R / "evaluation" / "bootstrap_test.csv")
verd = pd.read_csv(R / "evaluation" / "verdicts_test.csv")
halflife = int(rec["development_choices"]["ewma_halflife_days"])
win = pl.windows(daily, EXP); N = EXP["notional_usd"]
test_eps = [e for e in EXP["episodes"] if e["period"] == "test"]
fig_paths = {}
SYN = " [SYNTHETIC]" if MODE == "smoke" else ""

## Figure 1. The chain at a glance: refinancing state, exposure, hedge, outcome

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.3, 1.1, 1.3]})
ax = axes[0]
ax.plot(daily.index, daily["g"], color=pt.INK, lw=1.2)
ax.set_ylim(0, 1); ax.set_ylabel("share"); ax.set_title("1. Environment: estimated share of borrowers in the money" + SYN)
ax = axes[1]
vm = monthly[monthly["valid"]]
ax.scatter(vm["decision_date"], vm["D"], s=7, color="#9ec5f4", label="realized monthly duration")
wide = fc_test.pivot_table(index="decision_date", columns="model", values="forecast")
for s in ["S", "U", "H"]:
    st = pt.style(s); ax.plot(wide.index, wide[s], color=st["color"], ls=st["linestyle"], lw=1.5, drawstyle="steps-post", label=st["label"] + " forecast")
ax.set_ylim(min(-1.5, vm["D"].min() - 2.0), max(vm["D"].max(), 7) + 0.5)
ax.set_ylabel("years"); ax.set_title("2. Exposure: realized empirical duration and the hedges' forecasts"); ax.legend(fontsize=8, ncol=4, loc="lower center")
ax = axes[2]
hd = {}
for s in ["S", "U", "H"]:
    L = pd.read_parquet(R / "backtest" / f"ledger_test_base_{s}.parquet")
    hd[s] = L["h_held"] * daily.loc[L.index, "dur_h10"].shift(1).bfill()
    st = pt.style(s); ax.plot(hd[s].index, hd[s], color=st["color"], ls=st["linestyle"], lw=1.5, drawstyle="steps-post", label=st["label"])
ax.set_ylabel("years"); ax.set_title("3. Hedge: duration offset by the Treasury short"); ax.legend(fontsize=8, ncol=3, loc="upper left")
ax = axes[3]
cum = {}
for s in ["S", "U", "H"]:
    w = base[s]["weekly"]["gross_pnl"]; cum[s] = w.cumsum() * N / 1e6
    st = pt.style(s); ax.plot(cum[s].index, cum[s], color=st["color"], ls=st["linestyle"], lw=1.6, label=st["label"])
ax.axhline(0, color=pt.INK_2, lw=0.8)
ax.set_ylabel("$ millions per $100mm"); ax.set_title("4. Outcome: cumulative hedged excess P&L, test period (before costs)"); ax.legend(fontsize=8, ncol=3, loc="upper left")
for a in axes:
    pt.shade_episodes(a, test_eps, end_default=daily.index.max())
    a.axvline(pd.Timestamp(EXP["sample"]["test_start"] + "-01"), color=pt.INK_2, lw=1.0)
axes[0].text(pd.Timestamp(EXP["sample"]["test_start"] + "-01"), 0.97, " test period ->", transform=axes[0].get_xaxis_transform(), va="top", fontsize=8, color=pt.INK_2)
fig_paths["fig1_chain"] = pt.save_figure(fig, "fig1_chain_state_exposure_hedge_outcome", paths,
                                         data={"g": daily[["g"]], "forecasts": wide, "hedged_duration": pd.DataFrame(hd), "cum_pnl_usd_mm": pd.DataFrame(cum)}, mode=MODE)
plt.show()

## Figure 2. The main comparison with uncertainty

In [ ]:
prim = boot[boot["primary"]].copy()
labels = [f"{pt.STRATEGY_STYLE[a]['label']}  vs  {pt.STRATEGY_STYLE[b]['label']}" for a, b in zip(prim.challenger, prim.comparator)]
fig, ax = plt.subplots(figsize=(9.5, 3.6))
for i, (_, r) in enumerate(prim.iterrows()):
    col = pt.STRATEGY_STYLE[r["challenger"]]["color"]
    ax.errorbar(r["variance_reduction"], i, xerr=[[r["variance_reduction"] - r["vr_ci_low"]], [r["vr_ci_high"] - r["variance_reduction"]]],
                fmt="o", color=col, ms=6, lw=1.8, capsize=0)
    ax.text(r["vr_ci_high"], i, f"  {r['variance_reduction']:+.1%}  (Holm p = {r['p_holm']:.3f})", va="center", fontsize=8.5, color=pt.INK_2)
ax.axvline(0, color=pt.INK_2, lw=0.8)
ax.axvline(EXP["evaluation"]["practical_variance_reduction_vs_U"], color=pt.INK_2, lw=0.8, ls=":")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
ax.set_ylim(len(labels) - 0.5, -0.9)    # first comparison on top, with room above it for the target label
ax.text(EXP["evaluation"]["practical_variance_reduction_vs_U"], -0.62, " 5% practical target vs U", fontsize=8, color=pt.INK_2, va="center", ha="left")
ax.set_xlabel("test-period variance reduction of weekly hedged P&L (95% paired block-bootstrap interval)")
ax.set_title("Primary comparisons, base settings (monthly, 1-day delay, 2 bp)" + SYN)
fig_paths["fig2_main"] = pt.save_figure(fig, "fig2_primary_comparisons", paths, data=prim, mode=MODE)
plt.show()

## Figure 3. Risk against cost, and Figure 4. Stress episodes

In [ ]:
gm = pd.read_csv(R / "evaluation" / "grid_metrics_test.csv")
H_ = EXP["hedging"]
fr = gm[(gm["delay"] == H_["base_delay_days"]) & (gm["financing_bp"] == 0) & (gm["cost_bp"] == H_["base_cost_bp"]) & gm["strategy"].isin(["S", "U", "E", "H"])]
fig, ax = plt.subplots(figsize=(8, 4.8))
markers = {"monthly": "o", "quarterly": "s", "band_0.10": "^", "band_0.25": "v", "band_0.50": "D"}
for s in ["S", "U", "E", "H"]:
    d = fr[fr["strategy"] == s].sort_values("cost_bp_per_year"); st = pt.style(s)
    ax.scatter([], [], color=st["color"], marker="o", label=st["label"])
    for _, r in d.iterrows():
        ax.scatter(r["cost_bp_per_year"], r["vol_gross_ann_pct"], color=st["color"], marker=markers[r["rule"]], s=44, zorder=3)
for rule, mk in markers.items():
    ax.scatter([], [], color=pt.INK_2, marker=mk, label=rule)
ax.set_xlabel(f"annual cost drag at {H_['base_cost_bp']} bp per unit traded (bp/yr)"); ax.set_ylabel("annualized hedged volatility (%)")
ax.set_title("Risk against trading cost by rule" + SYN); ax.legend(fontsize=8, ncol=2)
fig_paths["fig3_frontier"] = pt.save_figure(fig, "fig3_risk_cost_frontier", paths, data=fr, mode=MODE)
plt.show()

shown = [e for e in test_eps if e["name"] in ("2016 election selloff", "2020 COVID rally and dislocation", "2022 rate shock", "2025-26 mortgage-rate rise")]
fig, axes = plt.subplots(1, len(shown), figsize=(3.2 * len(shown), 3.3), sharey=False)
axes = np.atleast_1d(axes)
ep_data = {}
for ax, e in zip(axes, shown):
    end = pd.Timestamp(e["end"]) if e["end"] else daily.index.max()
    for s in ["S", "U", "H"]:
        w = base[s]["weekly"]["gross_pnl"]; w = w[(w.index >= pd.Timestamp(e["start"])) & (w.index <= end + pd.Timedelta(days=6))]
        c = w.cumsum() * N / 1e6; ep_data[f"{e['name']}|{s}"] = c
        st = pt.style(s); ax.plot(c.index, c, color=st["color"], ls=st["linestyle"], lw=1.5, label=st["label"])
    ax.axhline(0, color=pt.INK_2, lw=0.8); ax.set_title(e["name"], fontsize=9.5); ax.tick_params(axis="x", labelrotation=45, labelsize=7.5)
axes[0].set_ylabel("cumulative hedged P&L ($mm per $100mm)"); axes[0].legend(fontsize=7.5)
fig.suptitle("Stress episodes in the test period" + SYN, x=0.01, ha="left", fontsize=12, fontweight="bold")
fig_paths["fig4_episodes"] = pt.save_figure(fig, "fig4_stress_episodes", paths, data=pd.DataFrame(ep_data), mode=MODE)
plt.show()

## Figure 5. Exposure against the refinancing state (descriptive, full sample)

Next month's realized duration against this month's share of borrowers in the money. The curve uses the economic model's coefficients from the last test decision (fitted on all data up to that date). This figure describes the mechanism. The forecasting and hedging tests are in Figure 2 and notebook 08.

In [ ]:
pairs = pd.DataFrame({"g": monthly["g"].shift(1), "D_next": monthly["D"], "valid": monthly["valid"]})
pairs["period"] = np.where(pairs.index <= win["development_end"], "development", "test")
pairs = pairs[pairs["valid"] & pairs["g"].notna()]
lastE = fc_test[fc_test["model"] == "E"].sort_values("decision_month").iloc[-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
dv, ts = pairs[pairs["period"] == "development"], pairs[pairs["period"] == "test"]
ax.scatter(dv["g"], dv["D_next"], s=20, facecolors="none", edgecolors="#2a78d6", label="development months (2007-2014)")
ax.scatter(ts["g"], ts["D_next"], s=20, color="#2a78d6", label="test months (2015 onward)")
xs = np.linspace(0, 1, 50)
ax.plot(xs, lastE["c0"] + lastE["c1"] * xs, color=pt.INK, lw=1.5, label=f"economic model at {lastE['decision_month']}: {lastE['c0']:.1f} {lastE['c1']:+.1f} x share")
ax.set_xlabel("share of borrowers in the money at month-end"); ax.set_ylabel("next month's realized duration (years)")
ax.set_title("Duration against refinancing conditions" + SYN); ax.legend(fontsize=8)
fig_paths["fig5_state"] = pt.save_figure(fig, "fig5_exposure_vs_state", paths, data=pairs, mode=MODE)
plt.show()

## Tables for the summary and deck

In [ ]:
tab = main.loc[["UNH", "S", "U", "E", "H", "P", "AR", "X"], ["vol_gross_ann_pct", "vol_gross_usd_mm", "var_reduction_vs_unhedged", "var_reduction_vs_S",
                                                         "var_reduction_vs_U", "residual_duration_years", "es95_weekly_net_bp",
                                                         "turnover_per_year", "cost_drag_bp_per_year", "cost_usd_per_year"]].copy()
tab.insert(0, "strategy_label", [pt.STRATEGY_STYLE[s]["label"] for s in tab.index])
pt.save_table(tab.round(4), "main_strategy_table", paths)
pt.save_table(verd.round(4), "cost_verdicts", paths, index=False)
for name in ["forecast_table_test", "robustness_test", "episodes_test", "regimes_test", "cost_grid_verdicts_test"]:
    f = R / "evaluation" / f"{name}.csv"
    if f.exists():
        pt.save_table(pd.read_csv(f), name, paths, index=False)
tab.round(3)

## Reconciliation: recompute the headline numbers from the weekly P&L file

The main table and bootstrap file were written by notebook 08. Here the same volatilities and variance reductions are recomputed directly from `results/backtest/weekly_test_base.parquet`, as an independent route to the same numbers.

In [ ]:
wk = pd.read_parquet(R / "backtest" / "weekly_test_base.parquet").pivot(index="week", columns="strategy", values="gross_pnl")
recon = []
for s in ["UNH", "S", "U", "E", "H"]:
    recon.append(("vol " + s, 100 * wk[s].std(ddof=1) * np.sqrt(52), main.loc[s, "vol_gross_ann_pct"]))
for a, b in [("H", "U"), ("U", "S"), ("E", "U"), ("H", "S")]:
    recon.append((f"VR {a} vs {b}", 1 - wk[a].var(ddof=1) / wk[b].var(ddof=1),
                  float(boot.loc[(boot.challenger == a) & (boot.comparator == b), "variance_reduction"].iloc[0])))
recon = pd.DataFrame(recon, columns=["quantity", "recomputed", "saved"])
recon["match"] = (recon["recomputed"] - recon["saved"]).abs() < 1e-9
display(recon)
assert recon["match"].all(), "Saved results and the weekly P&L file disagree"

## The claim ledger

Every number that may appear in the executive summary, slides, or talk is a row below, with its question, asset, sample, strategy, comparator, cost assumption, metric, units, value, uncertainty, source, and status. Status `synthetic_only` means the smoke run produced it and it must never be quoted. Simulation claims are always `synthetic_only` because they describe the procedure, not the market.

In [ ]:
ok = not rec.get("_post_freeze", False)
L = ClaimLedger()
def add(cid, q, asset, sample, strat, comp, cost, metric, units, result, unc, nb, fn, path, sim=False, available=True, note=""):
    st = status_for(path, "smoke" if (MODE == "smoke" or sim) else MODE, ok, available)
    L.add(claim_id=cid, question=q, asset=asset, sample=sample, strategy=strat, comparator=comp, cost_assumption=cost, metric=metric,
          units=units, result=result, uncertainty=unc, source_notebook=nb, source_function=fn, source_output_path=paths.rel(path), status=st, note=note)
test_s = f"{win['test_start']} to {win['test_end']}"
q1 = json.loads((R / "evaluation" / "q1_variation.json").read_text())
add("C01", "Q1", "MBB", f"{win['first_month']} to {win['test_end']}", "n/a", "n/a", "n/a", "5th-95th percentile of monthly realized duration", "years",
    f"{q1['p05_D']:.2f} to {q1['p95_D']:.2f}", f"median monthly se {q1['median_se']:.2f}", "08", "exposure.monthly_realized_durations", R / "evaluation" / "q1_variation.json")
add("C02", "Q1", "MBB", "full sample", "n/a", "constant duration", "n/a", "robust between-month SD of true duration (tau)", "years",
    f"{q1['full']['tau_robust']:.2f}", f"bootstrap p = {q1['full']['p_bootstrap']:.3g}", "08", "exposure.null_bootstrap_variation", R / "evaluation" / "q1_variation.json")
add("C03", "Q1", "MBB", test_s, "n/a", "constant duration", "n/a", "robust between-month SD of true duration (tau), test period", "years",
    f"{q1['test']['tau_robust']:.2f}", f"bootstrap p = {q1['test']['p_bootstrap']:.3g}", "08", "exposure.null_bootstrap_variation", R / "evaluation" / "q1_variation.json")
rel = json.loads((R / "exposure" / "state_relation_development.json").read_text())
add("C04", "mechanism", "MBB", "development months (descriptive)", "E fit", "n/a", "n/a", "locked-in vs all-in-the-money duration", "years",
    f"{rel['c0_locked_in_duration']:.2f} vs {rel['duration_if_all_in_money']:.2f}", f"corr {rel['corr']:+.2f}, {rel['share_proxy_months']:.0%} proxy months",
    "05", "models._wls", R / "exposure" / "state_relation_development.json", note="retrospective proxy before mid-2013")
ft = pd.read_csv(R / "evaluation" / "forecast_table_test.csv").set_index("model")
add("C05", "Q2", "MBB", test_s, "U, H, E, P, AR", "U", "n/a", "forecast RMSE of next-month duration", "years",
    "; ".join(f"{m} {ft.loc[m, 'rmse_years']:.2f}" for m in ["U", "H", "E", "P", "AR"] if m in ft.index), "common months", "08", "evaluation.forecast_table", R / "evaluation" / "forecast_table_test.csv")
ftests = json.loads((R / "evaluation" / "forecast_tests_test.json").read_text())
add("C06", "Q2", "MBB", test_s, "H", "U", "n/a", "Clark-West test of H over U", "p-value",
    f"{ftests['clark_west_H_vs_U']['p_one_sided']:.3f}", f"statistic {ftests['clark_west_H_vs_U']['cw_stat']:.2f}", "08", "evaluation.clark_west", R / "evaluation" / "forecast_tests_test.json")
add("C07", "Q3", "MBB", test_s, "S, U, E, H", "n/a", "gross", "annualized hedged volatility per $100mm", "$ millions",
    "; ".join(f"{s} {main.loc[s, 'vol_gross_usd_mm']:.2f}" for s in ["UNH", "S", "U", "E", "H"]), "see C08-C11", "08", "evaluation.strategy_summary", R / "evaluation" / "main_table_test.csv")
for cid, (a, b) in zip(["C08", "C09", "C10", "C11"], [("U", "S"), ("H", "U"), ("E", "U"), ("H", "S")]):
    r = boot[(boot.challenger == a) & (boot.comparator == b)].iloc[0]
    add(cid, "Q3", "MBB", test_s, a, b, "gross (costs in C12-C13)", "variance reduction of weekly hedged P&L", "fraction",
        f"{r['variance_reduction']:+.3f}", f"95% CI {r['vr_ci_low']:+.3f} to {r['vr_ci_high']:+.3f}; Holm p {r['p_holm']:.3f}", "08", "evaluation.paired_bootstrap", R / "evaluation" / "bootstrap_test.csv",
        note="headline comparison" if (a, b) == ("H", "U") else "")
for cid, (a, b) in zip(["C12", "C13"], [("H", "U"), ("U", "S")]):
    v = verd[(verd.challenger == a) & (verd.comparator == b)].iloc[0]
    add(cid, "Q4", "MBB", test_s, a, b, f"{EXP['hedging']['base_cost_bp']} bp per unit traded; hurdle IR {EXP['evaluation']['hurdle_information_ratio']}",
        "declared survives-costs verdict and break-even cost", "verdict; bp per unit traded",
        f"{v['verdict']}; net gain positive {v['net_gain_positive_when']}" + (f" ({v['breakeven_cost_bp_per_trade']:.1f})" if np.isfinite(v['breakeven_cost_bp_per_trade']) else ""),
        f"net gain {v['net_gain_bp_per_year']:+.2f} bp/yr", "08", "evaluation.cost_verdict", R / "evaluation" / "verdicts_test.csv")
add("C14", "Q4", "MBB", test_s, "S, U, H", "n/a", f"{EXP['hedging']['base_cost_bp']} bp", "turnover and annual cost", "x notional per year; $ per year",
    "; ".join(f"{s} {main.loc[s, 'turnover_per_year']:.2f}x, ${main.loc[s, 'cost_usd_per_year']:,.0f}" for s in ["S", "U", "H"]), "deterministic given trades", "07", "backtest.run_backtest", R / "evaluation" / "main_table_test.csv")
cg = pd.read_csv(R / "evaluation" / "cost_grid_verdicts_test.csv")
add("C15", "Q4", "MBB", test_s, "H", "U", "0 to 10 bp grid", "verdict across the cost grid", "verdict by cost",
    "; ".join(f"{int(c)} bp: {v}" for c, v in cg[cg.comparison == "H vs U"][["cost_bp", "verdict"]].values), "see C09", "08", "evaluation.cost_verdict", R / "evaluation" / "cost_grid_verdicts_test.csv")
ep = pd.read_csv(R / "evaluation" / "episodes_test.csv")
top = ep.loc[ep["share_of_total_HU_squared_pnl_reduction"].abs().idxmax()] if len(ep) else None
add("C16", "Q4", "MBB", test_s, "H", "U", "gross", "largest single-episode share of the H-vs-U squared-P&L reduction", "fraction",
    f"{top['episode']}: {top['share_of_total_HU_squared_pnl_reduction']:+.2f}" if top is not None else "n/a",
    f"leave-that-episode-out VR {top['vr_H_vs_U_leaving_out']:+.3f}" if top is not None else "", "08", "notebook 08 episode table", R / "evaluation" / "episodes_test.csv")
add("C17", "Q3", "MBB", test_s, "S, U, H", "zero", "gross", "residual duration of hedged P&L", "years",
    "; ".join(f"{s} {main.loc[s, 'residual_duration_years']:+.2f} (se {main.loc[s, 'residual_duration_se']:.2f})" for s in ["S", "U", "H"]), "HAC se", "08", "evaluation.residual_exposure", R / "evaluation" / "main_table_test.csv")
rb = pd.read_csv(R / "evaluation" / "robustness_test.csv")
base_v = rb["verdict_H_vs_U"].iloc[0]
add("C18", "robustness", "MBB", test_s, "H", "U", f"{EXP['hedging']['base_cost_bp']} bp", "declared checks with the same H-vs-U verdict as the base case", "count",
    f"{int((rb['verdict_H_vs_U'].iloc[1:] == base_v).sum())} of {len(rb) - 1}", "per-check intervals in the table", "08", "notebook 08 run_variant", R / "evaluation" / "robustness_test.csv")
xst = json.loads((R / "external" / "external_status.json").read_text()) if (R / "external" / "external_status.json").exists() else {}
vb = R / "external" / "vmbs_recipe_bootstrap.csv"
if vb.exists():
    r = pd.read_csv(vb).query("challenger == 'H' and comparator == 'U'").iloc[0]
    add("C19", "external", "VMBS", test_s, "H", "U", "gross", "variance reduction, recipe re-estimated on VMBS", "fraction", f"{r['variance_reduction']:+.3f}",
        f"95% CI {r['vr_ci_low']:+.3f} to {r['vr_ci_high']:+.3f}", "09", "evaluation.paired_bootstrap", vb)
else:
    add("C19", "external", "VMBS", test_s, "H", "U", "gross", "variance reduction, recipe re-estimated on VMBS", "fraction", "n/a", "", "09", "", vb,
        available=False, note=xst.get("VMBS", {}).get("reason", ""))
vc = R / "external" / "vmbs_coefficient_transfer_bootstrap.csv"
if vc.exists():
    r = pd.read_csv(vc).query("challenger == 'H' and comparator == 'U'").iloc[0]
    add("C20", "external", "VMBS", test_s, "H (MBB coefficients)", "U", "gross", "variance reduction, MBB coefficients applied unchanged", "fraction",
        f"{r['variance_reduction']:+.3f}", f"95% CI {r['vr_ci_low']:+.3f} to {r['vr_ci_high']:+.3f}", "09", "evaluation.paired_bootstrap", vc)
pm = xst.get("PMBS", {})
add("C21", "external (descriptive)", "PMBS vs MBB", "since PMBS launch", "n/a", "MBB", "n/a", "mean monthly empirical duration difference (PMBS minus MBB)", "years",
    f"{pm.get('mean_diff_years', float('nan')):+.2f}" if pm.get("mean_diff_years") is not None else "n/a",
    f"se {pm.get('mean_diff_se_years'):.2f}" if pm.get("mean_diff_se_years") else "", "09", "exposure.monthly_realized_durations", R / "external" / "pmbs_descriptive.csv",
    available=pm.get("status", "") not in ("unavailable", "excluded"),
    note=pm.get("reason", "") if pm.get("status") in ("unavailable", "excluded") else "descriptive, about two years, one regime; says nothing about PIMCO's trades")
pst = xst.get("PIMCO_predecessor", {})
pb = R / "external" / "pimco_predecessor_bootstrap.csv"
if pb.exists() and str(pst.get("status", "")).startswith("ran"):
    r = pd.read_csv(pb).query("challenger == 'H' and comparator == 'U'").iloc[0]
    add("C22", "external (PIMCO)", "PIMCO Mortgage-Backed Securities Fund before the ETF conversion (PTRIX)", f"{win['test_start']} to {pst.get('last_valid_month')}",
        "H", "U", "gross", "variance reduction, recipe re-estimated on the fund's own NAV history", "fraction", f"{r['variance_reduction']:+.3f}",
        f"95% CI {r['vr_ci_low']:+.3f} to {r['vr_ci_high']:+.3f}", "09", "evaluation.paired_bootstrap", pb, note=f"user-supplied NAV: {pst.get('file', '')}")
else:
    add("C22", "external (PIMCO)", "PIMCO Mortgage-Backed Securities Fund before the ETF conversion (PTRIX)", "test period up to the 2024 conversion", "H", "U", "gross",
        "variance reduction, recipe re-estimated on the fund's own NAV history", "fraction", pst.get("status", "n/a"), "", "09", "", R / "external" / "external_status.json",
        available=pst.get("status") not in (None, "unavailable"), note=pst.get("reason", ""))
look = json.loads((R / "forecasts" / "lookahead_check.json").read_text())
add("C23", "validation", "MBB", "test-period cut dates", "all feasible", "n/a", "n/a", "look-ahead perturbation checks passed", "count",
    f"{sum(r['passed'] for r in look['test_period_checks'])} of {len(look['test_period_checks'])}", "machine precision", "06", "pipeline.perturbation_check", R / "forecasts" / "lookahead_check.json")
pw = json.loads((R / "simulation" / "power_summary.json").read_text())
hu = next(c for c in pw["comparisons"] if c["comparison"] == "H_vs_U")
add("C24", "validation", "synthetic worlds", "simulation", "estimator", "truth", "n/a", "monthly duration estimator bias and 95% coverage", "years; share",
    f"bias {pw['recovery']['mean_error_years']:+.3f}; coverage {pw['recovery']['coverage_95']:.2f}", f"{pw['recovery']['months']} months", "04", "simulation.recovery_study", R / "simulation" / "power_summary.json", sim=True)
add("C25", "validation", "synthetic worlds", "simulation", "variation test", "nominal 5%", "n/a", "false-positive rate of the bootstrap variation test (naive chi-square in note)", "share",
    f"{pw['q_null_rejection_rate_5pct']:.2f}", f"naive chi-square {pw.get('q_null_rejection_rate_naive_chi2', float('nan')):.2f}", "04", "simulation.q_null_study", R / "simulation" / "power_summary.json", sim=True)
add("C26", "validation", "synthetic worlds", "simulation", "H", "U", f"{EXP['hedging']['base_cost_bp']} bp", "power of the survives-costs verdict at the state worlds' effect size; false-positive rate in unrelated worlds", "share; fraction",
    f"power {hu['state_share_survives_costs']:.2f} at mean VR {hu['state_mean_variance_reduction']:.3f}; false-positive rate {hu['null_share_survives_costs']:.2f}",
    f"{hu['n_state_worlds']} state and {hu['n_null_worlds']} unrelated worlds; median CI half-width {hu['state_median_ci_halfwidth']:.3f}", "04", "simulation.power_summary", R / "simulation" / "power_summary.json", sim=True)
oc = json.loads((R / "evaluation" / "outcome.json").read_text())
add("C27", "answer", "MBB", test_s, "H", "U", f"{EXP['hedging']['base_cost_bp']} bp", "pre-declared outcome classification", "label",
    oc["outcome"] + (" (inconclusive: a gain at the 5% target is not ruled out)" if oc.get("inconclusive") else
                     " (a gain at the 5% target is ruled out)" if oc.get("rules_out_practical_gain") else ""),
    f"VR {oc['variance_reduction_H_vs_U']:+.3f}, 95% CI {oc['ci_low']:+.3f} to {oc['ci_high']:+.3f}, Holm p {oc['p_holm_H_vs_U']:.3f}", "08",
    "evaluation.classify_outcome", R / "evaluation" / "outcome.json")
acc = pd.read_csv(R / "data_status" / "access_status.csv")
add("C28", "data", "all", "this run", "n/a", "n/a", "n/a", "access status of each source", "status",
    "; ".join(f"{d}: {s}" for d, s in acc[["dataset", "status"]].drop_duplicates("dataset", keep="last").values), "", "01", "sources.RawStore", R / "data_status" / "access_status.csv")
# PIMCO's GNMA fund (PDMIX) and the exposure comparison with the index (notebook 09, sections 4, 5, and 7)
gst = xst.get("PDMIX", {})
gb, gc = R / "external" / "pdmix_recipe_bootstrap.csv", R / "external" / "pdmix_coefficient_transfer_bootstrap.csv"
for cid, (a, b), f, method in [("C29", ("H", "U"), gb, "recipe re-estimated on PIMCO's GNMA fund"), ("C30", ("U", "S"), gb, "recipe re-estimated on PIMCO's GNMA fund"),
                               ("C31", ("H", "U"), gc, "MBB's refinancing coefficients applied unchanged")]:
    if f.exists():
        r = pd.read_csv(f).query("challenger == @a and comparator == @b").iloc[0]
        add(cid, "external (PIMCO)", "PDMIX (PIMCO GNMA fund)", test_s, a + (" (MBB coefficients)" if f == gc else ""), b, "gross", f"variance reduction, {method}", "fraction",
            f"{r['variance_reduction']:+.3f}", f"95% CI {r['vr_ci_low']:+.3f} to {r['vr_ci_high']:+.3f}", "09", "evaluation.paired_bootstrap", f,
            note="describes how the fund's value moved, not PIMCO's trades or skill")
    else:
        add(cid, "external (PIMCO)", "PDMIX (PIMCO GNMA fund)", test_s, a, b, "gross", f"variance reduction, {method}", "fraction", "n/a", "", "09", "", f,
            available=False, note=gst.get("reason", ""))
xc = R / "external" / "exposure_comparison_vs_mbb.csv"
xcd = pd.read_csv(xc) if xc.exists() else pd.DataFrame(columns=["fund", "sample", "status"])
for cid, fund, asset in [("C32", "PDMIX", "PDMIX (PIMCO GNMA fund) vs MBB"), ("C34", "PTRIX", "PTRIX (PIMCO MBS fund, user-supplied NAV) vs MBB")]:
    rr = xcd[(xcd["fund"] == fund) & (xcd["sample"] == "test period") & (xcd["status"] == "ok")]
    if len(rr):
        r = rr.iloc[0]
        add(cid, "external (PIMCO, descriptive)", asset, f"{r['first_month']} to {r['last_month']}", "n/a", "MBB", "n/a",
            "monthly empirical duration: correlation and mean difference (fund minus MBB)", "correlation; years",
            f"corr {r['corr']:.2f}; difference {r['mean_diff_years']:+.2f}", f"HAC se {r['mean_diff_se_years']:.2f}; {int(r['months'])} months", "09", "exposure.compare_exposure", xc)
        if fund == "PDMIX":
            add("C33", "external (PIMCO, descriptive)", asset, f"{r['first_month']} to {r['last_month']}", "n/a", "MBB", "n/a",
                "slope of the duration difference on the share of borrowers in the money at the start of the month", "years from g = 0 to g = 1",
                f"{r['slope_diff_years']:+.2f} (PDMIX {r['slope_x_years']:+.2f}, MBB {r['slope_ref_years']:+.2f})",
                f"95% CI {r['slope_diff_ci_low']:+.2f} to {r['slope_diff_ci_high']:+.2f}", "09", "exposure.compare_exposure", xc,
                note="mixes PIMCO's management with the Ginnie Mae pool; not a test")
    else:
        add(cid, "external (PIMCO, descriptive)", asset, "test period", "n/a", "MBB", "n/a", "monthly empirical duration: correlation and mean difference", "n/a",
            "n/a", "", "09", "", xc, available=False, note=(gst if fund == "PDMIX" else pst).get("reason", ""))
        if fund == "PDMIX":
            add("C33", "external (PIMCO, descriptive)", asset, "test period", "n/a", "MBB", "n/a", "slope of the duration difference on the refinancing share", "n/a",
                "n/a", "", "09", "", xc, available=False, note=gst.get("reason", ""))
ledger = L.frame()
written = L.save(R / "claims")
print("Claim statuses:", ledger["status"].value_counts().to_dict())
ledger[["claim_id", "question", "metric", "result", "uncertainty", "status"]]

## Limitations linked to evidence

In [ ]:
lim = pd.DataFrame([
    ("L01", "Synthetic constant-maturity Treasury instead of a futures contract (no basis, delivery option, or roll cost)", "results/evaluation/robustness_test.csv (financing spread, 5-year tenor)"),
    ("L02", "NMDB was published years after 2013, so the one-quarter lag is an availability approximation", "results/evaluation/robustness_test.csv (two-quarter lag)"),
    ("L03", "Pre-2013 incentive is a retrospective proxy", "results/forecasts/lookahead_check.json (development cut); robustness 'state model and hybrid use NMDB data only (no proxy)'"),
    ("L04", "ETF market prices can depart from NAV (March 2020)", "results/evaluation/robustness_test.csv (excluding March-April 2020)"),
    ("L05", "Empirical duration mixes prepayment effects with spread moves that co-move with rates", "results/evaluation/convexity_test.csv; docs/assumptions_limitations.md"),
    ("L06", "Few independent refinancing cycles in the test period", "results/simulation/power_summary.json (power and false-positive rate); results/evaluation/outcome.json (interval upper end); results/evaluation/episodes_test.csv"),
    ("L07", "Monthly error bars from about 21 days understate sampling error", "results/simulation/power_summary.json (coverage); variation test uses the bootstrap"),
    ("L08", "Aggregate stock note rate is a proxy for MBB borrowers' incentive", "results/state/state_info.json; robustness 'All Mortgages' market"),
    ("L09", "Cost levels are declared assumptions, not measured costs", "results/evaluation/cost_grid_verdicts_test.csv; break-even costs in verdicts_test.csv"),
    ("L10", "VMBS is not an independent market. PIMCO's GNMA fund mixes active management with a different mortgage pool, so its differences from MBB cannot be attributed to either. The PIMCO MBS fund's pre-2024 history needs a user-supplied file", "results/external/external_status.json; results/external/exposure_comparison_vs_mbb.csv"),
    ("L11", "Freddie Mac changed the PMMS method on 2022-11-17", "not separately tested (docs/assumptions_limitations.md)"),
    ("L12", "A duration hedge removes only first-order rate risk", "results/evaluation/main_table_test.csv (ex-post diagnostic X); results/evaluation/convexity_test.csv"),
    ("L13", "Yields are fixed around 3:30 pm ET while MBB closes at 4 pm", "results/exposure/lead_lag_development.json"),
    ("L14", "Mutual-fund NAVs (PIMCO's GNMA fund, PTRIX) come from evaluated prices, which can be smoother than traded prices", "results/external/exposure_comparison_vs_mbb.csv (correlation with MBB's monthly durations)"),
], columns=["limitation_id", "limitation", "evidence"])
lim.to_csv(R / "claims" / "limitations_evidence.csv", index=False)
lim

## Run manifest and return bundle

In [ ]:
import platform, hashlib
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
versions = {}
for name in ["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "pyarrow", "yfinance", "nbformat", "nbclient"]:
    try:
        versions[name] = __import__(name).__version__
    except Exception:
        versions[name] = "not installed"
manifest = {"mode": MODE, "python": platform.python_version(), "platform": platform.platform(), "packages": versions,
            "freeze_created_utc": rec["created_utc"], "experiment_sha256_frozen": rec["experiment_sha256"], "experiment_sha256_now": fz.experiment_hash(cfg),
            "code_sha256_frozen": rec["code_sha256"], "code_sha256_now": fz.code_hash(paths.src), "post_freeze": bool(rec.get("_post_freeze", False)),
            "config_file_sha256": sha(paths.root / "config" / "project.yaml"),
            "figures": sorted(p.name for p in paths.figures.glob("*.png")), "tables": sorted(p.name for p in paths.tables.glob("*.csv")),
            "claim_statuses": ledger["status"].value_counts().to_dict()}
(R / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
from mbshedge.bundle import make_bundle
b = make_bundle(paths, MODE)
print(f"Return bundle: {paths.rel(b['zip'])} ({b['n_files']} files, {b['bytes'] / 1e6:.1f} MB). run_project.py rebuilds it at the very end so the final run log is included.")

## Checklist for the communication stage

Before writing the executive summary, slides, and talk, work through these steps in order:

1. **Reconcile the bundle**: check `BUNDLE_MANIFEST.csv` hashes, read `logs/run_log_*` and `logs/pytest_*`, and confirm the freeze record and `test_evaluation_log.csv` (how many times the test was evaluated, and whether any evaluation was post-freeze).
2. **Check errors and accounting**: confirm every core notebook finished, the reconciliation checks in notebook 07 passed, and the look-ahead checks in notebook 06 passed. Recompute the main table's volatilities and variance reductions from `results/backtest/weekly_test_base.parquet` and compare them with `results/evaluation/main_table_test.csv`.
3. **Regenerate the main results from saved evidence only**, without refitting any model and without changing the frozen experiment.
4. **Write from the claim ledger**: every number in the executive summary, slides, and talk must be a `verified_empirical` row (or be labeled as simulation evidence). Use `docs/outcome_interpretations.md` for the outcome that notebook 08 selected.
5. **Use the 2025 templates** in `communication/templates_2025/` as described in `docs/presentation_template_notes.md`: same slide dimensions, fonts, colors, and title placement, adapted to this project's story and to the Fall 2026 requirements (problem, method, findings, limitations, and what was learned).